In [11]:
using LowLevelFEM, LinearAlgebra#, SparseArrays

In [12]:
openGeometry("cylinder.geo")
#openPreProcessor()

In [13]:
mat = Material("shell", E=2e5, ν=0.3)

U = Field([mat], type=:VectorField, dim=3, field=:u)
Φ = Field([mat], type=:VectorField, dim=3, field=:φ);

In [14]:
n = normalVector(U, "shell")
tξ, tη = tangentVectors(U, "shell");

In [15]:
p = 1n;

In [16]:
showElementResults(n, name="n")
showElementResults(tξ, name="tξ")
showElementResults(tη, name="tη");

In [17]:
n = [n[1], n[2], n[3]]
tξ = [tξ[1], tξ[2], tξ[3]]
tη = [tη[1], tη[2], tη[3]];

In [18]:
# kron(tξ, tη)' ⋅ Grad(U) gives tξᵀ(∇U)tη.

A_u = vcat(kron(tξ,tξ)',
           kron(tη,tη)',
           (kron(tξ,tη) + kron(tη,tξ))',
           zeros(3,9),
           kron(n,tη)',
           kron(n,tξ)')

A_φ = vcat(zeros(3,9),
           kron(tη,tξ)',
           -kron(tξ,tη)',
           (kron(tη,tη) - kron(tξ,tξ))',
           zeros(2,9))

G_φ = vcat(zeros(6,3), -tξ', tη');

In [19]:
b = 0.12
k_s = 4/5

C = D(:PlaneStress, mat)

D1 = b*C

D2 = b^3 / 12 * C

D3 = k_s * b * mat.μ * Matrix{Float64}(I, 2, 2)

# Locking miatt a nyírást csak redukáltan integráljuk.
B_mb = A_u[1:6, :] ⋅ Grad(U) + A_φ[1:6, :] ⋅ Grad(Φ)
B_s  = A_u[7:8, :] ⋅ Grad(U) + G_φ[7:8, :] ⋅ Φ
D_mb = cat(D1, D2; dims=(1, 2))

K = ∫(B_mb' ⋅ D_mb ⋅ B_mb; Γ="shell") +
    ∫(B_s'  ⋅ D3   ⋅ B_s;  Γ="shell", gauss=:reduced)

K[:,:]

14346×14346 SparseArrays.SparseMatrixCSC{Float64, Int64} with 963654 stored entries:
⎡⣿⣿⣿⣯⣿⣿⣿⣿⣷⢰⣶⣷⢲⣄⣒⣂⡀⣶⣶⣾⣿⣽⣿⣿⣽⢿⣿⣿⡆⢰⣾⡖⣶⣐⣂⣂⢰⣶⣶⣿⎤
⎢⡿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢙⣛⠉⠉⠉⠉⠉⠉⣻⣾⣿⣿⣿⣿⣿⣿⣷⣿⣿⣿⣛⣛⠉⠉⠉⠉⠉⠉⣷⣿⣿⣿⣿⣿⣿⣿⣾⣿⣿⣟⎥
⎢⢼⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣾⡿⠀⠀⠀⠀⠀⠀⣟⣿⣿⣿⣿⢿⡿⣿⣿⣿⡿⣿⎥
⎢⠘⢶⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣷⣧⣿⣿⣷⣿⠛⣶⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣶⣿⢿⡿⣿⎥
⎢⠸⢸⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⠽⣿⣿⣿⣿⣿⣿⣽⠿⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⡷⎥
⎢⢠⣬⠀⠀⠀⠀⠀⠀⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢠⡍⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⡿⣿⣿⣿⣿⣿⣗⎥
⎢⣸⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣽⣿⣟⣿⣿⣿⣿⣿⣾⣇⠀⠀⠀⠀⠀⠀⡿⣿⣿⣵⣿⣽⢿⡿⣿⣿⣿⣿⎥
⎢⣟⣿⣿⣿⣿⣿⣿⣿⣿⢸⣾⡿⢻⣤⣿⣧⡄⠶⠾⢿⣿⣿⣿⣿⣿⣿⣿⣿⡇⢸⣾⠟⣿⣼⣧⣧⠰⠶⠾⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣷⣟⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢈⣉⠉⠉⠉⠉⠁⠉⣽⣿⣿⣽⣿⣿⣿⣿⣿⣿⣿⣯⣉⣉⠉⠉⠁⠉⠈⠉⣿⣿⣿⣽⣿⣿⣿⣿⣿⣿⣿⣯⎥
⎢⢺⠿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢟⣿⣾⠟⠀⠀⠀⠀⠀⠀⣟⣿⣿⣿⣿⣿⡿⡿⣿⣿⡿⣿⎥
⎢⢘⢻⠀⠀⠀⠀⠀⠀⣿⣿⣿⣟⣿⣿⣿⣷⣿⡿⣟⣿⣛⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣾⣻⡻⣟⣿⎥
⎢⠨⢸⠀⠀⠀⠀⠀⠀⣿⣿⣿⣯⢻⣿⣿⣿⣿⣿⣿⡷⠭⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⡯⣻⣿⣿⣿⣿⣿⣿⡶⎥
⎢⢰⣶⠀⠀⠀⠀⠀⠀⣾⣿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⢰⡆⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⡺⣿⣿⣿⣿⣿⣏⎥
⎣⣼⣿⠀⠀⠀⠀⠀⠀⣿⢿⣿⣯⣿⣯⢿⡿⢿⢿⣿⣿⣾⣧⠀⠀⠀⠀⠀⠀⡿⣿⣿⣯⣿⣽⢻⡿⡿⢿⣿⣿⎦

In [20]:
A_d = (kron(tξ,tη) - kron(tη,tξ))' / 2
B_d = A_d ⋅ Grad(U) + n' ⋅ Φ

α = 1e-4
μ = mat.μ
Kd = ∫(B_d' ⋅ (α * b * μ) ⋅ B_d; Γ="shell")
Kd[:,:]

14346×14346 SparseArrays.SparseMatrixCSC{Float64, Int64} with 962609 stored entries:
⎡⣿⣿⣿⣯⣿⣿⣿⣿⣷⢰⣶⣷⢲⣄⣒⣂⡀⣶⣶⣾⣿⣽⣿⣿⣽⢿⣿⣿⡆⢰⣾⡖⣶⣐⣂⣂⢰⣶⣶⣿⎤
⎢⡿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢙⣛⠉⠉⠉⠉⠉⠉⣻⣾⣿⣿⣿⣿⣿⣿⣷⣿⣿⣿⣛⣛⠉⠉⠉⠉⠉⠉⣷⣿⣿⣿⣿⣿⣿⣿⣾⣿⣿⣟⎥
⎢⢼⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣾⡿⠀⠀⠀⠀⠀⠀⣟⣿⣿⣿⣿⢿⡿⣿⣿⣿⡿⣿⎥
⎢⠘⢶⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣷⣧⣿⣿⣷⣿⠛⣶⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣶⣿⢿⡿⣿⎥
⎢⠸⢸⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⠽⣿⣿⣿⣿⣿⣿⣽⠿⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⡷⎥
⎢⢠⣬⠀⠀⠀⠀⠀⠀⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢠⡍⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⡿⣿⣿⣿⣿⣿⣗⎥
⎢⣸⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣽⣿⣟⣿⣿⣿⣿⣿⣾⣇⠀⠀⠀⠀⠀⠀⡿⣿⣿⣵⣿⣽⢿⡿⣿⣿⣿⣿⎥
⎢⣟⣿⣿⣿⣿⣿⣿⣿⣿⢸⣾⡿⢻⣤⣿⣧⡄⠶⠾⢿⣿⣿⣿⣿⣿⣿⣿⣿⡇⢸⣾⠟⣿⣼⣧⣧⠰⠶⠾⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣷⣟⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢈⣉⠉⠉⠉⠉⠁⠉⣽⣿⣿⣽⣿⣿⣿⣿⣿⣿⣿⣯⣉⣉⠉⠉⠁⠉⠈⠉⣿⣿⣿⣽⣿⣿⣿⣿⣿⣿⣿⣯⎥
⎢⢺⠿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢟⣿⣾⠟⠀⠀⠀⠀⠀⠀⣟⣿⣿⣿⣿⣿⡿⡿⣿⣿⡿⣿⎥
⎢⢘⢻⠀⠀⠀⠀⠀⠀⣿⣿⣿⣟⣿⣿⣿⣷⣿⡿⣟⣿⣛⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣾⣻⡻⣟⣿⎥
⎢⠨⢸⠀⠀⠀⠀⠀⠀⣿⣿⣿⣯⢻⣿⣿⣿⣿⣿⣿⡷⠭⣿⠀⠀⠀⠀⠀⠀⣿⣿⣿⡯⣻⣿⣿⣿⣿⣿⣿⡶⎥
⎢⢰⣶⠀⠀⠀⠀⠀⠀⣾⣿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⢰⡆⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⡺⣿⣿⣿⣿⣿⣏⎥
⎣⣼⣿⠀⠀⠀⠀⠀⠀⣿⢿⣿⣯⣿⣯⢿⡿⢿⢿⣿⣿⣾⣧⠀⠀⠀⠀⠀⠀⡿⣿⣿⣯⣿⣽⢻⡿⡿⢿⣿⣿⎦

In [21]:
fu = ∫(U ⋅ p; Γ="shell")
fφ = ∫(Φ ⋅ [0.0, 0.0, 0.0]; Γ="shell")
f = SystemVector([fu, fφ]);

In [22]:
txy = tangentVector(U, "xy")
tyz = tangentVector(U, "yz")
tzx = tangentVector(U, "zx")

showElementResults(txy, name="t xy")
showElementResults(tyz, name="t yz")
showElementResults(tzx, name="t zx")

bc_uxy = BoundaryCondition("xy"; field=U, uz=0)
bc_uyz = BoundaryCondition("yz"; field=U, ux=0)
bc_uzx = BoundaryCondition("zx"; field=U, uy=0)
bc_φ = BoundaryCondition("zx"; field=U, ux=0, uy=0, uz=0);

In [23]:
u, φ = solveField(K + Kd, f, support=[bc_uxy, bc_uyz, bc_uzx]);

In [24]:
showDoFResults(u, name="u", visible=true, factor=10);

In [25]:
n = normalVector(U, "shell")
tξ, tη = tangentVectors(U, "shell")
φe = nodesToElements(φ)

gu, gφ = ∇(u), ∇(φe)
uξ, uη = gu * tξ, gu * tη
φξ, φη = gφ * tξ, gφ * tη

εm = [tξ ⋅ uξ, tη ⋅ uη, (tξ ⋅ uη) + (tη ⋅ uξ)]
κb = [tη ⋅ φξ, -(tξ ⋅ φη), (tη ⋅ φη) - (tξ ⋅ φξ)]

Nξ, Nη, Nξη = D1 * εm
Mξ, Mη, Mξη = D2 * κb

γξ = (n ⋅ uξ) + (φe ⋅ tη)
γη = (n ⋅ uη) - (φe ⋅ tξ)

# On this affine Q1 mesh, the nodal mean gives the element-center value.
center(s) = ScalarField(
    [repeat(sum(a; dims=1) / size(a, 1), size(a, 1), 1) for a in s.A],
    [;;], s.t, s.numElem, s.nsteps, s.type, s.model
)

Qη, Qξ = D3 * [center(γη), center(γξ)];

In [26]:
showElementResults(Nξ;  name="N_xi")
showElementResults(Nη;  name="N_eta")
showElementResults(Nξη; name="N_xi_eta")

showElementResults(Mξ;  name="M_xi")
showElementResults(Mη;  name="M_eta")
showElementResults(Mξη; name="M_xi_eta")

showElementResults(Qξ;  name="Q_xi")
showElementResults(Qη;  name="Q_eta");

In [27]:
N = [Nξ, Nη, Nξη]
M = [Mξ, Mη, Mξη]

σ_top = N / b + (6 / b^2) * M
σ_mid = N / b
σ_bot = N / b - (6 / b^2) * M

vonMises(σ) = sqrt(σ[1]^2 - σ[1] * σ[2] + σ[2]^2 + 3 * σ[3]^2)

showElementResults(vonMises(σ_top); name="In-plane von Mises - top (+n)")
showElementResults(vonMises(σ_mid); name="In-plane von Mises - midsurface")
showElementResults(vonMises(σ_bot); name="In-plane von Mises - bottom (-n)");

In [28]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
